# Why Deep, Not Fat?
### An interactive tour from activation pieces to 2^K, and the fish-and-bear's-paw of depth

A neural network with **one** hidden layer can already approximate *any* continuous function (the Universal Approximation Theorem). So why does the field obsess over **depth**? If width is enough in principle, why stack layers?

This notebook answers that question by *building the argument you can run*:

1. **Sum of pieces** -- a curve is a constant plus a sum of activation 'pieces'; a one-hidden-layer net is just that sum. (C03-C08)
2. **Universal Approximation** -- a wide shallow net fits anything, which turns the question from *can it?* to *how efficiently?* (C09-C11)
3. **The 2^K construction** -- the lecture's centerpiece: K layers of just 2 ReLUs each build 2^K linear pieces. (C12-C16)
4. **Matched-budget experiment** -- at *equal* parameter count, a thin-and-tall net beats a fat-and-short one on complex, regular targets. (C17-C19)
5. **The |H| tradeoff** -- depth reaches huge expressivity with an exponentially smaller hypothesis set, so it lands a low loss *and* a small generalization gap. (C20-C22)

The punchline, tying back to the ML-Basics generalization lecture: depth lets us **have both the fish and the bear's paw** -- a low achievable loss and a small generalization gap at the same time.

> Everything runs on synthetic 1-D functions, on CPU, in seconds. Use **Runtime -> Run all**, then play with the sliders.

In [ ]:
# C02 -- Setup: imports, reproducibility, plotting defaults
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# ipywidgets -- degrade gracefully if unavailable
try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider, FloatSlider, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    interact = IntSlider = FloatSlider = Dropdown = None
    WIDGETS_AVAILABLE = False

# Reproducibility
SEED = 0
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

# PyTorch -- optional (used by the deep-vs-shallow experiment); numpy fallback otherwise
try:
    import torch
    torch.manual_seed(SEED)
    TORCH_AVAILABLE = True
except Exception:
    torch = None
    TORCH_AVAILABLE = False

# Consistent plotting defaults
plot_defaults = {
    'figure.figsize': (7, 4.5),
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
    'lines.linewidth': 2.0,
}
plt.rcParams.update(plot_defaults)

# --- sanity checks ---
assert rng is not None and hasattr(rng, 'standard_normal')
assert plt.rcParams['figure.figsize'][0] == 7
assert isinstance(WIDGETS_AVAILABLE, bool) and isinstance(TORCH_AVAILABLE, bool)

print('numpy       ', np.__version__)
print('matplotlib  ', matplotlib.__version__)
print('ipywidgets  ', 'available' if WIDGETS_AVAILABLE else 'NOT available (static fallbacks used)')
print('torch       ', torch.__version__ if TORCH_AVAILABLE else 'NOT available (numpy fallback used)')
print('SEED        ', SEED)

## 1. The atomic building block: one activation 'piece'

Every function we build is made of repeated copies of a single shape:

> **piece(x) = c * act(b + w * x)**

Three knobs reshape that one piece:

- **b (shift)** -- slides the piece left/right. The interesting action happens near `b + w*x = 0`, i.e. `x = -b/w`.
- **w (slope / steepness)** -- how fast the piece turns on. Large `|w|` makes a sharp step; small `|w|` a gentle ramp. The sign of `w` flips its direction.
- **c (height)** -- scales the piece up or down (a negative `c` flips it vertically).

We will use three activations:

- **sigmoid:** `c * sigmoid(b + w*x)` -- a smooth S-shaped step.
- **hard-sigmoid:** a piecewise-linear S (slope 0.25, clamped to [0,1]) -- the 'cartoon' version of the sigmoid.
- **ReLU:** `c * max(0, b + w*x)` -- a hinge with a kink at `x = -b/w`.

A useful fact (p.6-8): **two opposite ReLUs add up to a hard-sigmoid step**. So ReLU hinges and sigmoid steps are two views of the same Lego brick -- which is why summing them can reconstruct any wiggly curve.

In [ ]:
# C04 -- Activation primitives and the single piece c * act(b + w*x)
def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

def relu(z):
    return np.maximum(0.0, np.asarray(z, dtype=float))

def hard_sigmoid(z):
    # piecewise-linear sigmoid: slope 0.25, clamped at z = +/- 2 -> [0, 1]
    return np.clip(0.5 + 0.25 * np.asarray(z, dtype=float), 0.0, 1.0)

_ACTIVATIONS = {'sigmoid': sigmoid, 'relu': relu, 'hard_sigmoid': hard_sigmoid}

def activation_piece(x, b, w, c, kind='sigmoid'):
    '''One activation piece: c * act(b + w*x). b shifts, w sets slope, c sets height.'''
    if kind not in _ACTIVATIONS:
        raise ValueError('unknown activation kind: ' + str(kind))
    x = np.asarray(x, dtype=float)
    return c * _ACTIVATIONS[kind](b + w * x)

# Reference shapes for the three activations
_xx = np.linspace(-5, 5, 400)
single_piece_figure, _axes = plt.subplots(1, 3, figsize=(13, 4))
for _ax, _kind in zip(_axes, ['sigmoid', 'relu', 'hard_sigmoid']):
    _ax.plot(_xx, activation_piece(_xx, b=0.0, w=1.0, c=1.0, kind=_kind))
    _ax.set_title(_kind + '  (b=0, w=1, c=1)')
    _ax.set_xlabel('x')
    _ax.set_ylabel('piece(x)')
single_piece_figure.suptitle('The three reference activation pieces', y=1.03)
plt.tight_layout()
plt.show()

# --- sanity checks ---
assert np.isclose(sigmoid(0), 0.5)
assert np.allclose(relu([-1, 2]), [0, 2])
assert np.allclose(hard_sigmoid([-10, 10]), [0, 1])
assert np.isclose(activation_piece(0, 0, 1, 3, 'sigmoid'), 1.5)
try:
    activation_piece(0, 0, 1, 1, 'bogus')
    raise AssertionError('expected ValueError')
except ValueError:
    pass
print('C04 OK -- activations and activation_piece ready')

In [ ]:
# C05 -- Interactive single-piece explorer
_x_piece = np.linspace(-5, 5, 400)

def _plot_piece(b=0.0, w=1.0, c=1.0, kind='sigmoid'):
    y = activation_piece(_x_piece, b, w, c, kind)
    plt.figure()
    plt.plot(_x_piece, y, color='C0')
    plt.axhline(0, color='0.7', lw=1)
    if kind == 'relu' and w != 0:
        kink = -b / w
        if _x_piece.min() <= kink <= _x_piece.max():
            plt.axvline(kink, color='C3', ls='--', lw=1, label='kink at x = -b/w')
            plt.legend()
    pad = 0.1 * (abs(c) + 1.0)
    plt.ylim(float(np.min(y)) - pad, float(np.max(y)) + pad)
    plt.title('c*act(b + w*x):  b=%.2f, w=%.2f, c=%.2f, kind=%s' % (b, w, c, kind))
    plt.xlabel('x')
    plt.ylabel('piece(x)')
    plt.show()

if WIDGETS_AVAILABLE:
    piece_explorer_widget = interact(
        _plot_piece,
        b=FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='b (shift)'),
        w=FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='w (slope)'),
        c=FloatSlider(value=1.0, min=-3.0, max=3.0, step=0.1, description='c (height)'),
        kind=Dropdown(options=['sigmoid', 'relu', 'hard_sigmoid'], value='sigmoid', description='activation'),
    )
else:
    piece_explorer_widget = None
    print('ipywidgets unavailable -- showing a static example (b=0, w=2, c=1, relu).')
    _plot_piece(0.0, 2.0, 1.0, 'relu')

# --- sanity checks ---
_plot_piece(0, 2, 1, 'relu')
assert (not WIDGETS_AVAILABLE) or (piece_explorer_widget is not None)
print('C05 OK -- piece explorer ready')

## Summing pieces = a one-hidden-layer network

Take a target curve. Approximate it as a **constant plus a sum of pieces**:

> **f(x) = b0 + sum_i c_i * act(b_i + w_i * x)**

That expression *is* a network with one hidden layer: each hidden neuron is one piece, its weight/bias set `w_i, b_i`, and the output weight sets the height `c_i`.

A piecewise-linear curve with many segments needs many hinges -- so **more pieces means a finer fit**. In the next cell we fix evenly-spaced ReLU hinges and solve only the heights by least squares (fast, deterministic, no training noise), then watch the error shrink as we add pieces.

In [ ]:
# C07 -- One hidden layer as a sum of pieces (least-squares heights on fixed knots)
W_FIX = 1.0
x_grid = np.linspace(0.0, 1.0, 200)

def target_fn(x):
    x = np.asarray(x, dtype=float)
    return np.sin(2.0 * np.pi * x) + 0.4 * np.sin(5.0 * np.pi * x)

def fit_pieces(N, kind='relu'):
    N = int(N)
    if N < 1:
        raise ValueError('N must be >= 1 (need at least one piece)')
    knots = np.linspace(0.0, 1.0, N, endpoint=False)
    cols = [np.ones_like(x_grid)]                      # explicit constant column
    for t in knots:
        cols.append(activation_piece(x_grid, b=-W_FIX * t, w=W_FIX, c=1.0, kind=kind))
    Phi = np.stack(cols, axis=1)                       # shape (200, N+1)
    y = target_fn(x_grid)
    heights, _res, _rank, _sv = np.linalg.lstsq(Phi, y, rcond=None)
    approx = Phi @ heights
    mse = float(np.mean((approx - y) ** 2))
    return {'N': N, 'knots': knots, 'Phi': Phi, 'heights': heights,
            'approx': approx, 'mse': mse, 'kind': kind}

approx_with_pieces = fit_pieces(20, kind='relu')

approximation_figure, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4.5))
_y = target_fn(x_grid)
axL.plot(x_grid, _y, 'k', lw=2.5, label='target f(x)')
axL.plot(x_grid, approx_with_pieces['approx'], 'C1--', lw=2, label='approx (N=20)')
axL.set_title('Target vs sum-of-pieces  (MSE=%.4g)' % approx_with_pieces['mse'])
axL.set_xlabel('x')
axL.set_ylabel('f(x)')
axL.legend()

h = approx_with_pieces['heights']
axR.axhline(h[0], color='0.6', ls=':', label='constant term')
for j, t in enumerate(approx_with_pieces['knots']):
    axR.plot(x_grid, h[j + 1] * activation_piece(x_grid, b=-W_FIX * t, w=W_FIX, c=1.0, kind='relu'),
             lw=1, alpha=0.7)
axR.set_title('Individual scaled ReLU pieces (their sum = approx)')
axR.set_xlabel('x')
axR.set_ylabel('piece height')
axR.legend()
plt.tight_layout()
plt.show()

# --- sanity checks ---
assert approx_with_pieces['Phi'].shape == (200, 21)
assert fit_pieces(40)['mse'] <= fit_pieces(4)['mse']
assert np.all(np.isfinite(approx_with_pieces['approx']))
assert approx_with_pieces['mse'] < 0.05
print('C07 OK -- MSE(N=20) = %.4g' % approx_with_pieces['mse'])

In [ ]:
# C08 -- Interactive sum-of-pieces: more pieces -> better fit
_y_target = target_fn(x_grid)

def _plot_approx(N=8):
    N = int(N)
    res = fit_pieces(N, kind='relu')
    plt.figure()
    plt.plot(x_grid, _y_target, 'k', lw=2.5, label='target f(x)')
    plt.plot(x_grid, res['approx'], 'C1--', lw=2, label='approx (N=%d)' % N)
    plt.title('N = %d ReLU pieces    MSE = %.5f' % (N, res['mse']))
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.show()
    print('N=%2d pieces -> MSE = %.6f' % (N, res['mse']))

if WIDGETS_AVAILABLE:
    pieces_count_widget = interact(_plot_approx, N=IntSlider(value=8, min=1, max=40, step=1, description='N pieces'))
else:
    pieces_count_widget = None
    print('ipywidgets unavailable -- showing N = 2, 8, 20 statically.')
    for _N in (2, 8, 20):
        _plot_approx(_N)

# --- sanity checks ---
_plot_approx(5)
assert (not WIDGETS_AVAILABLE) or (pieces_count_widget is not None)
assert fit_pieces(2)['mse'] > fit_pieces(20)['mse']
print('C08 OK -- pieces-count explorer ready')

## 2. The Universal Approximation Theorem (UAT)

> **A feed-forward net with a single hidden layer, given enough hidden units, can approximate any continuous function on a closed interval to arbitrary accuracy.** (p.4, 14)

So a shallow-but-wide net is already a universal approximator. The previous demo made this tangible: pile on pieces and the fit gets as good as you like.

This reframes the whole debate. The question is **not**:

> *Can a shallow network represent this function?* -- the answer is **yes**.

The real question is:

> *How efficiently?* How many neurons does it cost?

Depth is an **efficiency** argument, not a possibility argument. The next sections show that some functions a deep net captures with a handful of neurons would cost a shallow net an **exponential** number.

In [ ]:
# C10 -- Universal Approximation, empirically: a wide shallow net fits anything
_W_MAX = 128
_feat_rng = np.random.default_rng(SEED + 1)
_kinks = _feat_rng.uniform(0.0, 1.0, _W_MAX)
_slopes = _feat_rng.uniform(0.5, 8.0, _W_MAX) * _feat_rng.choice([-1.0, 1.0], size=_W_MAX)

def shallow_fit_routine(W, kind='relu'):
    W = int(W)
    if W < 1:
        raise ValueError('W must be >= 1')
    W = min(W, _W_MAX)
    cols = [np.ones_like(x_grid)]
    for j in range(W):
        cols.append(activation_piece(x_grid, b=-_slopes[j] * _kinks[j], w=_slopes[j], c=1.0, kind=kind))
    Phi = np.stack(cols, axis=1)
    y = target_fn(x_grid)
    heights, _r, _k, _s = np.linalg.lstsq(Phi, y, rcond=None)
    approx = Phi @ heights
    return {'W': W, 'approx': approx, 'mse': float(np.mean((approx - y) ** 2)),
            'Phi': Phi, 'heights': heights}

widths = [2, 4, 8, 16, 32, 64]
width_sweep_curve = np.array([[W, shallow_fit_routine(W)['mse']] for W in widths], dtype=float)

uat_figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
_y = target_fn(x_grid)
for W, style in zip((2, 8, 64), ('C0:', 'C1--', 'C2-')):
    axes[0].plot(x_grid, shallow_fit_routine(W)['approx'], style, label='W=%d' % W)
axes[0].plot(x_grid, _y, 'k', lw=2.5, label='target')
axes[0].set_title('Shallow fit at increasing width W')
axes[0].set_xlabel('x')
axes[0].legend()

_eps = 1e-12
axes[1].semilogy(width_sweep_curve[:, 0], width_sweep_curve[:, 1] + _eps, 'o-')
axes[1].set_title('Approximation error vs width (log-y)')
axes[1].set_xlabel('hidden width W (neurons)')
axes[1].set_ylabel('MSE')
plt.tight_layout()
plt.show()

# --- sanity checks ---
assert width_sweep_curve.shape == (len(widths), 2)
assert width_sweep_curve[-1, 1] <= width_sweep_curve[0, 1]
assert np.all(np.isfinite(width_sweep_curve))
assert np.isfinite(shallow_fit_routine(8)['mse'])
print('C10 OK -- MSE: W=2 -> %.4g, W=64 -> %.4g' % (width_sweep_curve[0, 1], width_sweep_curve[-1, 1]))

In [ ]:
# C11 -- Interactive UAT explorer: fit improves with width, but at what neuron cost?
def _plot_shallow(W=8):
    W = int(W)
    res = shallow_fit_routine(W)
    plt.figure()
    plt.plot(x_grid, target_fn(x_grid), 'k', lw=2.5, label='target')
    plt.plot(x_grid, res['approx'], 'C2--', lw=2, label='shallow fit (W=%d)' % res['W'])
    plt.title('Single hidden layer, width W = %d    MSE = %.5f' % (res['W'], res['mse']))
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.show()
    print('width W = %d neurons -> MSE = %.6f' % (res['W'], res['mse']))

if WIDGETS_AVAILABLE:
    uat_explorer_widget = interact(_plot_shallow, W=IntSlider(value=8, min=1, max=128, step=1, description='width W'))
else:
    uat_explorer_widget = None
    print('ipywidgets unavailable -- showing W = 4, 16, 64 statically.')
    for _W in (4, 16, 64):
        _plot_shallow(_W)

# --- sanity checks ---
_plot_shallow(16)
assert (not WIDGETS_AVAILABLE) or (uat_explorer_widget is not None)
assert shallow_fit_routine(16)['W'] == 16
print('C11 OK -- UAT width explorer ready')

## 3. The centerpiece: K layers x 2 neurons -> 2^K pieces (p.19-22)

Here is the exact construction from the lecture. One hidden layer of **just two ReLU neurons** computes a triangular 'mountain' map on [0,1]:

> **m(x) = 2*relu(x) - 4*relu(x - 0.5)**

- input weights +1 and +1, biases 0 and -0.5,
- output weights +2 and -4.

Check the corners: `m(0)=0`, `m(0.5)=1`, `m(1)=0` -- a tent rising then falling, mapping [0,1] back onto [0,1].

Now **compose** it. Feeding a tent through another tent folds the line in half again, doubling the number of linear pieces each time:

> **K layers of 2 ReLUs  ->  2^K linear pieces.**

The summary contrast that powers the whole lecture:

| network | neurons | pieces |
|---|---|---|
| **Deep** | 2K (linear in K) | 2^K |
| **Shallow** | 2^K (exponential) | 2^K |

Same expressivity. Exponentially different cost.

In [ ]:
# C13 -- The exact mountain tent map: m(x) = 2*relu(x) - 4*relu(x - 0.5)  (p.19-22 integer weights)
def mountain_map(x):
    x = np.asarray(x, dtype=float)
    return 2.0 * relu(x) - 4.0 * relu(x - 0.5)

_xt = np.linspace(0.0, 1.0, 500)
triangle_figure = plt.figure()
plt.plot(_xt, mountain_map(_xt), color='C0', lw=2.5)
for px, py in [(0.0, 0.0), (0.5, 1.0), (1.0, 0.0)]:
    plt.plot(px, py, 'o', color='C3')
    plt.annotate('(%.1f, %.1f)' % (px, py), (px, py), textcoords='offset points', xytext=(6, 6))
plt.title('One 2-ReLU layer = a triangle mapping [0,1] -> [0,1]')
plt.xlabel('x')
plt.ylabel('m(x)')
plt.ylim(-0.05, 1.1)
plt.show()

# --- sanity checks ---
assert np.allclose(mountain_map([0, 0.5, 1]), [0, 1, 0])
_chk = mountain_map(_xt)
assert _chk.min() >= -1e-9 and _chk.max() <= 1 + 1e-9
assert np.isclose(mountain_map(0.25), 0.5) and np.isclose(mountain_map(0.75), 0.5)
print('C13 OK -- mountain_map ready')

In [ ]:
# C14 -- Compose the tent K times -> 2^K linear pieces
def compose_layers(x, K):
    K = int(K)
    if K < 0:
        raise ValueError('K must be >= 0')
    y = np.asarray(x, dtype=float)
    for _ in range(K):
        y = mountain_map(y)
    return y

def _count_segments(y, tol=1e-9):
    d = np.diff(y)
    s = np.sign(d)
    s = s[np.abs(d) > tol]
    if s.size == 0:
        return 1
    return int(np.sum(s[1:] != s[:-1]) + 1)

x_dense = np.linspace(0.0, 1.0, 4000)
deep_function_K = {K: compose_layers(x_dense, K) for K in (1, 2, 3)}
piece_count_table = [(K, 2 ** K) for K in (1, 2, 3)]

two_K_figure, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, K in zip(axes, (1, 2, 3)):
    ax.plot(x_dense, deep_function_K[K], color='C0', lw=1.5)
    ax.set_title('K=%d  ->  %d pieces (measured %d)' % (K, 2 ** K, _count_segments(deep_function_K[K])))
    ax.set_xlabel('x')
    ax.set_ylabel('composed f(x)')
two_K_figure.suptitle('K layers of 2 ReLUs produce 2^K linear pieces', y=1.04)
plt.tight_layout()
plt.show()

print('K   2^K   measured pieces')
for K, p in piece_count_table:
    print('%d   %3d   %d' % (K, p, _count_segments(deep_function_K[K])))

# --- sanity checks ---
assert np.allclose(compose_layers([0, 1], 3), [0, 0])
for K in (1, 2, 3):
    assert _count_segments(deep_function_K[K]) == 2 ** K
assert piece_count_table == [(1, 2), (2, 4), (3, 8)]
assert deep_function_K[3].min() >= -1e-9 and deep_function_K[3].max() <= 1 + 1e-9
print('C14 OK -- compose_layers verified to give 2^K pieces')

In [ ]:
# C15 -- Interactive depth explorer: expressivity explodes as 2^K, neurons grow only as 2K
_x_depth = np.linspace(0.0, 1.0, 4000)

def _plot_depth(K=3):
    K = int(np.clip(K, 1, 8))
    y = compose_layers(_x_depth, K)
    deep_neurons = 2 * K
    shallow_neurons = 2 ** K
    plt.figure(figsize=(9, 4))
    plt.plot(_x_depth, y, color='C0', lw=1.2)
    plt.title('K = %d layers  ->  2^%d = %d linear pieces' % (K, K, 2 ** K))
    plt.xlabel('x')
    plt.ylabel('composed f(x)')
    plt.ylim(-0.05, 1.1)
    plt.show()
    print('K=%d:  deep needs 2K = %d neurons   vs   shallow needs 2^K = %d neurons' % (K, deep_neurons, shallow_neurons))

if WIDGETS_AVAILABLE:
    depth_explorer_widget = interact(_plot_depth, K=IntSlider(value=3, min=1, max=8, step=1, description='depth K'))
else:
    depth_explorer_widget = None
    print('ipywidgets unavailable -- showing K = 1, 3, 6 statically.')
    for _K in (1, 3, 6):
        _plot_depth(_K)

# --- sanity checks ---
_plot_depth(4)
assert (not WIDGETS_AVAILABLE) or (depth_explorer_widget is not None)
assert 2 * 5 == 10 and 2 ** 5 == 32
print('C15 OK -- depth explorer ready')

## (Optional aside) 2^K shows up everywhere: parity, modularity, paper-folding

*This is an enrichment detour -- skip it and the runnable narrative still holds.*

The exponential payoff of composition is not special to neural nets:

- **Parity / XOR chains.** Computing the parity of `d` bits with a *two-layer* logic circuit needs O(2^d) gates, but a *multi-layer* chain of XNOR gates does it with O(d) gates. Same exponential-vs-linear gap as deep-vs-shallow nets (p.15-16).
- **Modular programming.** 'Don't put everything in main()' -- factoring code into reusable functions that call each other is exactly composition. A flat program re-implements the same sub-routine over and over (p.17).
- **Fold-then-cut.** Fold a paper K times, make one cut, unfold: you get 2^K holes. Each fold (a layer) doubles the structure (p.17).

All three are the same idea: **reuse via composition buys exponential structure for linear effort.**

## 4. Does depth actually win at equal cost? A matched-budget experiment

The theory says deep reaches 2^K pieces with only 2K neurons. Let us **test it under a fair budget**: give a deep thin-and-tall net and a shallow fat-and-short net (approximately) the **same number of parameters**, train both, and compare.

Targets:

- the **2^K sawtooth** from our `compose_layers` map (complex but *regular* -- built from repeated structure), and
- **y = x^2** (smooth) for contrast.

This is a tiny synthetic stand-in for the lecture's speech-recognition tables (Seide et al., 2011, p.10-12), where deeper networks beat shallow ones at matched parameter counts on real word-error-rate benchmarks. Here you can run it yourself in a few CPU seconds.

In [ ]:
# C18 -- Matched-parameter budget: thin-and-tall (deep) vs fat-and-short (shallow)

# ---- targets ----
K_TARGET = 4
x_train = np.linspace(0.0, 1.0, 256).reshape(-1, 1)
y_saw = compose_layers(x_train.ravel(), K_TARGET).reshape(-1, 1)   # 2^4 = 16-piece sawtooth
y_sq = x_train ** 2                                                 # y = x^2 (smooth), also available

# ---- numpy MLP (fallback when torch is absent) ----
class _NumpyMLP:
    def __init__(self, sizes, seed=0):
        r = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.W = [r.standard_normal((sizes[i], sizes[i + 1])) * np.sqrt(2.0 / sizes[i])
                  for i in range(len(sizes) - 1)]
        self.b = [np.zeros(sizes[i + 1]) for i in range(len(sizes) - 1)]
    def forward(self, X, cache=False):
        a = np.asarray(X, dtype=float)
        acts = [a]
        zs = []
        L = len(self.W)
        for i in range(L):
            z = a @ self.W[i] + self.b[i]
            a = np.maximum(0.0, z) if i < L - 1 else z
            zs.append(z)
            acts.append(a)
        if cache:
            return a, zs, acts
        return a

def _params_for_sizes(sizes):
    return int(sum(sizes[i] * sizes[i + 1] + sizes[i + 1] for i in range(len(sizes) - 1)))

def _match_shallow_width(deep_widths, in_dim=1, out_dim=1):
    deep_params = _params_for_sizes([in_dim] + list(deep_widths) + [out_dim])
    width = max(1, int(round((deep_params - out_dim) / (in_dim + 1 + out_dim))))
    return width

def make_mlp(widths, in_dim=1, out_dim=1):
    sizes = [in_dim] + list(widths) + [out_dim]
    if TORCH_AVAILABLE:
        layers = []
        for i in range(len(sizes) - 1):
            layers.append(torch.nn.Linear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:
                layers.append(torch.nn.ReLU())
        return torch.nn.Sequential(*layers)
    return _NumpyMLP(sizes, seed=SEED)

def count_params(model):
    if TORCH_AVAILABLE and isinstance(model, torch.nn.Module):
        return int(sum(p.numel() for p in model.parameters()))
    return int(sum(w.size for w in model.W) + sum(b.size for b in model.b))

def train_model(model, x, y, epochs=1500, lr=1e-2):
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    report_every = max(1, epochs // 3)
    if TORCH_AVAILABLE and isinstance(model, torch.nn.Module):
        Xt = torch.tensor(x, dtype=torch.float32)
        Yt = torch.tensor(y, dtype=torch.float32)
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        lossfn = torch.nn.MSELoss()
        losses = []
        for ep in range(epochs):
            opt.zero_grad()
            out = model(Xt)
            loss = lossfn(out, Yt)
            if not torch.isfinite(loss):
                raise FloatingPointError('non-finite loss during torch training')
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))
            if (ep + 1) % report_every == 0:
                print('    epoch %4d/%d  loss=%.5f' % (ep + 1, epochs, losses[-1]))
        with torch.no_grad():
            pred = model(Xt).numpy()
        return {'losses': losses, 'pred': pred, 'backend': 'torch'}
    # ---- numpy Adam fallback ----
    n = x.shape[0]
    L = len(model.W)
    mW = [np.zeros_like(w) for w in model.W]
    vW = [np.zeros_like(w) for w in model.W]
    mb = [np.zeros_like(bb) for bb in model.b]
    vb = [np.zeros_like(bb) for bb in model.b]
    b1, b2, eps = 0.9, 0.999, 1e-8
    losses = []
    for t in range(1, epochs + 1):
        out, zs, acts = model.forward(x, cache=True)
        diff = out - y
        loss = float(np.mean(diff ** 2))
        if not np.isfinite(loss):
            raise FloatingPointError('non-finite loss during numpy training')
        losses.append(loss)
        delta = (2.0 / n) * diff
        dW = [None] * L
        db = [None] * L
        for i in range(L - 1, -1, -1):
            dW[i] = acts[i].T @ delta
            db[i] = delta.sum(axis=0)
            if i > 0:
                delta = (delta @ model.W[i].T) * (zs[i - 1] > 0)
        for i in range(L):
            mW[i] = b1 * mW[i] + (1 - b1) * dW[i]
            vW[i] = b2 * vW[i] + (1 - b2) * dW[i] ** 2
            mb[i] = b1 * mb[i] + (1 - b1) * db[i]
            vb[i] = b2 * vb[i] + (1 - b2) * db[i] ** 2
            model.W[i] -= lr * (mW[i] / (1 - b1 ** t)) / (np.sqrt(vW[i] / (1 - b2 ** t)) + eps)
            model.b[i] -= lr * (mb[i] / (1 - b1 ** t)) / (np.sqrt(vb[i] / (1 - b2 ** t)) + eps)
        if t % report_every == 0:
            print('    epoch %4d/%d  loss=%.5f' % (t, epochs, losses[-1]))
    return {'losses': losses, 'pred': model.forward(x), 'backend': 'numpy'}

# ---- build matched-budget pair ----
deep_widths = [8, 8, 8, 8]
shallow_widths = [_match_shallow_width(deep_widths)]
print('backend:', 'torch' if TORCH_AVAILABLE else 'numpy')
print('deep widths   ', deep_widths)
print('shallow widths', shallow_widths)

_EPOCHS = 1200
print('training deep (thin and tall) ...')
deep_model = make_mlp(deep_widths)
deep_result = train_model(deep_model, x_train, y_saw, epochs=_EPOCHS, lr=1e-2)
print('training shallow (fat and short) ...')
shallow_model = make_mlp(shallow_widths)
shallow_result = train_model(shallow_model, x_train, y_saw, epochs=_EPOCHS, lr=1e-2)

deep_params = count_params(deep_model)
shallow_params = count_params(shallow_model)
print('deep params =', deep_params, ' shallow params =', shallow_params)

# ---- plot ----
deep_vs_shallow_figure, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.5))
axA.plot(x_train.ravel(), y_saw.ravel(), 'k', lw=2.5, label='target (2^4 sawtooth)')
axA.plot(x_train.ravel(), np.asarray(deep_result['pred']).ravel(), 'C0--', lw=2,
         label='deep %s (%dp)' % (deep_widths, deep_params))
axA.plot(x_train.ravel(), np.asarray(shallow_result['pred']).ravel(), 'C3:', lw=2,
         label='shallow %s (%dp)' % (shallow_widths, shallow_params))
axA.set_title('Fit at matched parameter budget')
axA.set_xlabel('x')
axA.legend(fontsize=8)
axB.semilogy(deep_result['losses'], 'C0', label='deep (final %.4g)' % deep_result['losses'][-1])
axB.semilogy(shallow_result['losses'], 'C3', label='shallow (final %.4g)' % shallow_result['losses'][-1])
axB.set_title('Training loss')
axB.set_xlabel('epoch')
axB.set_ylabel('MSE')
axB.legend(fontsize=8)
plt.tight_layout()
plt.show()

# --- sanity checks ---
assert abs(deep_params - shallow_params) <= 0.05 * deep_params
assert np.isfinite(deep_result['losses'][-1]) and np.isfinite(shallow_result['losses'][-1])
assert deep_result['losses'][-1] <= deep_result['losses'][0]
assert shallow_result['losses'][-1] <= shallow_result['losses'][0]
if deep_result['losses'][-1] > shallow_result['losses'][-1]:
    print('WARN: deep did not beat shallow this run (try more epochs or re-seed).')
print('C18 OK -- deep final %.5f vs shallow final %.5f' % (deep_result['losses'][-1], shallow_result['losses'][-1]))

In [ ]:
# C19 -- Interactive matched-budget explorer: deep's edge widens as the target gets more complex
_RESULT_CACHE = {}
_EXPLORE_EPOCHS = 500

def run_for_K(K):
    K = int(K)
    if K in _RESULT_CACHE:
        return _RESULT_CACHE[K]
    xt = np.linspace(0.0, 1.0, 256).reshape(-1, 1)
    yt = compose_layers(xt.ravel(), K).reshape(-1, 1)
    dm = make_mlp(deep_widths)
    sm = make_mlp(shallow_widths)
    dr = train_model(dm, xt, yt, epochs=_EXPLORE_EPOCHS, lr=1e-2)
    sr = train_model(sm, xt, yt, epochs=_EXPLORE_EPOCHS, lr=1e-2)
    out = {'K': K,
           'deep_loss': float(dr['losses'][-1]),
           'shallow_loss': float(sr['losses'][-1]),
           'deep_params': count_params(dm),
           'shallow_params': count_params(sm)}
    _RESULT_CACHE[K] = out
    return out

def _plot_budget(K=4):
    K = int(K)
    r = run_for_K(K)
    plt.figure(figsize=(6, 4))
    bars = plt.bar(['deep (thin and tall)', 'shallow (fat and short)'],
                   [r['deep_loss'], r['shallow_loss']], color=['C0', 'C3'])
    plt.ylabel('final train MSE')
    plt.title('K=%d target (%d pieces), matched params %d vs %d'
              % (K, 2 ** K, r['deep_params'], r['shallow_params']))
    for bar, v in zip(bars, [r['deep_loss'], r['shallow_loss']]):
        plt.text(bar.get_x() + bar.get_width() / 2, v, '%.4g' % v, ha='center', va='bottom', fontsize=9)
    plt.show()
    print('K=%d: deep MSE=%.5f  shallow MSE=%.5f' % (K, r['deep_loss'], r['shallow_loss']))

if WIDGETS_AVAILABLE:
    deep_vs_shallow_widget = interact(_plot_budget, K=IntSlider(value=4, min=1, max=6, step=1, description='target K'))
else:
    deep_vs_shallow_widget = None
    print('ipywidgets unavailable -- showing K = 2, 4, 6 statically.')
    for _K in (2, 4, 6):
        _plot_budget(_K)

# --- sanity checks ---
_r3 = run_for_K(3)
assert np.isfinite(_r3['deep_loss']) and np.isfinite(_r3['shallow_loss'])
assert run_for_K(3) is _RESULT_CACHE[3]
assert (not WIDGETS_AVAILABLE) or (deep_vs_shallow_widget is not None)
print('C19 OK -- matched-budget explorer ready')

## 5. The thesis: depth shrinks effective |H| while keeping loss low

Recall the ML-Basics tradeoff. A bigger hypothesis set H lowers the *best achievable* loss, but **widens the generalization gap** (the model can also fit noise). Picking model complexity used to feel like choosing one or the other.

Depth dodges the dilemma. To reach **2^K linear pieces**:

- a **shallow** net needs ~2^K neurons -- a *huge* hypothesis set H, hence a wide gap;
- a **deep** net needs only **2K** neurons -- an exponentially *smaller* H for the **same expressivity**.

So depth secures a **low achievable loss** (it can express the rich target) **and** a **small generalization gap** (its effective |H| stays tiny) at the same time.

That is the lecture's punchline (p.2, 22, 24): deep learning lets you **have both the fish and the bear's paw** -- the two things the bias/variance tradeoff said you had to choose between.

In [ ]:
# C21 -- The complexity tradeoff: linear deep cost vs exponential shallow cost
Ks = np.arange(1, 13)
neuron_count_deep = 2 * Ks
neuron_count_shallow = 2 ** Ks

complexity_figure, (axA, axB) = plt.subplots(1, 2, figsize=(13, 4.5))
axA.semilogy(Ks, neuron_count_shallow, 'C3o-', label='shallow: 2^K neurons')
axA.semilogy(Ks, neuron_count_deep, 'C0s-', label='deep: 2K neurons')
axA.set_title('Neurons needed for 2^K pieces')
axA.set_xlabel('K (depth)')
axA.set_ylabel('neuron count (log scale)')
axA.legend()

# schematic |H| vs achievable-loss / generalization-gap (NOT fitted data)
_h = np.linspace(0.1, 10, 200)
axB.plot(_h, 1.0 / _h, 'C2', label='ideal achievable loss (down)')
axB.plot(_h, 0.05 * _h, 'C1', label='generalization gap (up)')
axB.axvline(2.0, color='C0', ls='--', lw=1.2, label='deep: small |H|')
axB.axvline(8.0, color='C3', ls=':', lw=1.2, label='shallow: large |H|')
axB.set_title('schematic (not fitted): |H| vs loss and gap')
axB.set_xlabel('model complexity |H| (neuron/param proxy)')
axB.set_ylabel('loss')
axB.legend(fontsize=8)
plt.tight_layout()
plt.show()

# --- sanity checks ---
assert np.array_equal(neuron_count_deep, 2 * Ks)
assert 2 ** 8 == 256
assert np.all(neuron_count_shallow >= neuron_count_deep) and np.all(neuron_count_shallow[2:] > neuron_count_deep[2:])
assert np.all(neuron_count_shallow > 0)
print('C21 OK -- complexity tradeoff figure rendered')

In [ ]:
# C22 -- Interactive tradeoff explorer: feel the exponential gap in |H|
def _plot_tradeoff(K=6):
    K = int(np.clip(K, 1, 12))
    deep = 2 * K
    shallow = 2 ** K
    ratio = shallow / deep
    plt.figure(figsize=(8, 4.5))
    plt.semilogy(Ks, neuron_count_shallow, 'C3o-', label='shallow: 2^K')
    plt.semilogy(Ks, neuron_count_deep, 'C0s-', label='deep: 2K')
    plt.scatter([K], [shallow], s=120, color='C3', zorder=5, edgecolor='k')
    plt.scatter([K], [deep], s=120, color='C0', zorder=5, edgecolor='k')
    plt.title('K=%d: deep %d vs shallow %d neurons (ratio %.1fx)' % (K, deep, shallow, ratio))
    plt.xlabel('K (depth)')
    plt.ylabel('neurons for 2^K pieces (log scale)')
    plt.legend()
    plt.annotate('depth keeps |H| small while reaching 2^K pieces: both fish and bear paw',
                 xy=(0.04, 0.93), xycoords='axes fraction', fontsize=9, color='C0')
    plt.show()
    print('K=%d: deep = 2K = %d,  shallow = 2^K = %d,  ratio = %.1fx' % (K, deep, shallow, ratio))

if WIDGETS_AVAILABLE:
    tradeoff_explorer_widget = interact(_plot_tradeoff, K=IntSlider(value=6, min=1, max=12, step=1, description='K'))
else:
    tradeoff_explorer_widget = None
    print('ipywidgets unavailable -- showing K = 3, 6, 10 statically.')
    for _K in (3, 6, 10):
        _plot_tradeoff(_K)

# --- sanity checks ---
_plot_tradeoff(6)
assert (not WIDGETS_AVAILABLE) or (tradeoff_explorer_widget is not None)
assert 2 * 10 == 20 and 2 ** 10 == 1024 and abs((2 ** 10) / (2 * 10) - 51.2) < 1e-9
print('C22 OK -- tradeoff explorer ready')

## Wrap-up: why deep, not fat?

We built one continuous argument, each section answering a learning objective:

1. **Sum of pieces** (C03-C08) -- a curve = constant + sum of activation pieces; a one-hidden-layer net is that sum, and more pieces -> better fit.
2. **Universal Approximation** (C09-C11) -- a wide shallow net fits anything, so the question becomes *how efficiently?*, not *whether*.
3. **The 2^K construction** (C12-C16) -- K layers of 2 ReLUs compose into 2^K linear pieces; expressivity grows exponentially while neurons grow linearly (2K).
4. **Matched-budget experiment** (C17-C19) -- at equal parameter count, deep beats shallow on complex, regular targets, and its edge widens with complexity.
5. **The |H| tradeoff** (C20-C22) -- depth reaches that expressivity with an exponentially smaller hypothesis set, so it lands a low loss *and* a small generalization gap.

**The answer:** depth is not about *can* a network fit the data -- a fat one already can. It is about *efficiency*. By reusing structure through composition, deep networks express complex, regular functions with exponentially fewer neurons, keeping the effective |H| small. That is how deep learning gets to **have both the fish and the bear's paw**: a low loss and a small gap together.

**Where to go next:** convolutional and recurrent architectures (composition with structure), optimization (why depth is *trainable* in practice), and generalization theory beyond neuron-counting (VC dimension, Rademacher complexity).